
# Ad/Commercial Pilot — 실제 데이터 병행 산출 및 검증

`docs/AD_COMMERCIAL_PILOT.md`가 설명하듯, 원본 "Ad/Commercial Index"(20개 업종 키워드매칭,
`ad_commercial_index_v7.json`)는 이번 세션에 존재하지 않는다. 남아있는
`scripts/indices_csv/build_ad_commercial_index_csv.py`도 업종 목록 자체를 하드코딩하지 않고
원본 JSON에서 읽어오는 구조라, 20개 업종이 무엇이었는지는 전혀 복구할 수 없다.

대신 이번 세션에 실제로 복구된 `data/v6_r22_snapshot/fandom_scores_v6.json`에는 K=8/M=6 LDA
메타요인 분해 결과인 `factor_share`가 100개 팬덤 전원에 대해 이미 계산되어 있고, 그중 하나가
`docs/LDA_V6_V7_TECHNICAL_SPECIFICATION.md` 4장이 정의하는 F2 "브랜드·상업형(광고·앰버서더)"
(키워드: 브랜드·앰버서더·발탁·광고·모델·수상·활동·캠페인·홍보대사·음악)다.

이 노트북은 그 값을 **원본의 재현이 아니라, 같은 질문(광고·브랜드 성격 비중)에 답하는 독립적인
병행 산출물**로 취급해, 1절에서 데이터 무결성을 검증하고 2~4절에서 실제로 순위화·분포 분석한다.


In [1]:

import json
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 140)

DATA_DIR = Path("../data/v6_r22_snapshot")
FACTOR = "브랜드·상업형(광고·앰버서더)"

with open(DATA_DIR / "fandom_scores_v6.json", encoding="utf-8") as f:
    scores = json.load(f)

print(f"fandom_scores_v6.json: {len(scores)}개 팬덤")
print("샘플(BTS) factor_share:", json.dumps(scores[0]["factor_share"], ensure_ascii=False, indent=2))


fandom_scores_v6.json: 100개 팬덤
샘플(BTS) factor_share: {
  "소비력형(초동·판매·앨범)": 0.27,
  "현장경제형(콘서트·투어·매진)": 0.292,
  "브랜드·상업형(광고·앰버서더)": 0.0472,
  "소비력형(초동·판매·앨범)·데뷔형": 0.1614,
  "결속형(팬클럽·기부·커뮤니티)": 0.0855,
  "미디어노출형(방송·조회수)": 0.1439
}



## 1. 데이터 무결성 검증

`factor_share`는 팬덤별로 6개 메타요인에 대한 비중을 나타내므로, 그 합이 1.0이어야 한다(LDA
문서-토픽 혼합비중을 메타요인 단위로 합산한 것이므로). 100개 팬덤 전원에 대해 재확인한다. 또한
`dominant_factor` 필드가 실제로 `factor_share` 중 최댓값과 일치하는지도 함께 재계산해
대조한다.


In [2]:

sum_mismatch = []
dominant_mismatch = []

for d in scores:
    fname = d["fandom"]
    shares = d["factor_share"]
    s = sum(shares.values())
    if abs(s - 1.0) > 0.001:
        sum_mismatch.append((fname, round(s, 4)))

    recomputed_dominant = max(shares, key=shares.get)
    if recomputed_dominant != d["dominant_factor"]:
        dominant_mismatch.append((fname, recomputed_dominant, d["dominant_factor"]))

print(f"factor_share 합 != 1.0 인 팬덤 수: {len(sum_mismatch)} / {len(scores)}")
if sum_mismatch:
    print("  불일치 목록:", sum_mismatch[:10])

print(f"dominant_factor 재계산 불일치 팬덤 수: {len(dominant_mismatch)} / {len(scores)}")
if dominant_mismatch:
    print("  불일치 목록:", dominant_mismatch[:10])


factor_share 합 != 1.0 인 팬덤 수: 0 / 100
dominant_factor 재계산 불일치 팬덤 수: 0 / 100



## 2. "브랜드·상업형(광고·앰버서더)" 비중 — 원본 "광고비중"에 대응하는 병행 지표

원본 스키마의 "광고비중"(`ad_share = n_ad_bullets / n_total_bullets`)에 대응하는 값으로,
F2 메타요인 비중을 100개 팬덤 전체에 대해 산출해 내림차순 정렬한다. 원본의 "대표업종"에는
`dominant_factor`(6개 메타요인 중 최댓값)를 함께 표시하되, 이는 20개 업종이 아니라 6개
메타요인 중 하나를 가리킨다는 점을 컬럼명에서부터 구분해 표기한다.


In [3]:

rows = []
for d in scores:
    rows.append({
        "팬덤": d["fandom"],
        "구분": d["category"],
        "근거문장수(activity)": d["activity"],
        "브랜드·상업형 비중": round(d["factor_share"].get(FACTOR, 0.0), 4),
        "대표 메타요인(6개 중)": d["dominant_factor"],
        "대표 메타요인이 브랜드·상업형인가": d["dominant_factor"] == FACTOR,
    })

ad_df = pd.DataFrame(rows).sort_values("브랜드·상업형 비중", ascending=False).reset_index(drop=True)
ad_df.index = ad_df.index + 1
ad_df.head(15)


,팬덤,구분,근거문장수(activity),브랜드·상업형 비중,대표 메타요인(6개 중),대표 메타요인이 브랜드·상업형인가
1,사이먼도미닉,힙합,17,0.2553,브랜드·상업형(광고·앰버서더),True
2,(여자)아이들,K-pop 걸그룹,64,0.2261,소비력형(초동·판매·앨범)·데뷔형,False
3,장윤정,트로트,41,0.2056,현장경제형(콘서트·투어·매진),False
4,정동원,트로트,63,0.2013,결속형(팬클럽·기부·커뮤니티),False
5,한로로,솔로,33,0.1861,현장경제형(콘서트·투어·매진),False
6,SEVENTEEN,K-pop 보이그룹,113,0.1778,현장경제형(콘서트·투어·매진),False
7,지코,힙합,67,0.1753,현장경제형(콘서트·투어·매진),False
8,김호중,트로트,65,0.1742,결속형(팬클럽·기부·커뮤니티),False
9,창모,힙합,22,0.1693,현장경제형(콘서트·투어·매진),False
10,BE'O,힙합,28,0.1575,현장경제형(콘서트·투어·매진),False



## 3. 대표 메타요인이 "브랜드·상업형"인 팬덤은 몇 개인가

원본의 "대표업종"처럼 이 메타요인이 그 팬덤의 **가장 큰** 성격 요인인 경우가 실제로 얼마나
되는지 확인한다. 대부분의 팬덤에게는 이 요인이 "구성 비중 중 하나"일 뿐 "대표" 성격은 아닐
가능성이 높다는 점을 문서에서 예고했으므로, 그대로 재확인한다.


In [4]:

n_dominant = (ad_df["대표 메타요인이 브랜드·상업형인가"]).sum()
print(f"대표 메타요인(dominant_factor)이 '브랜드·상업형(광고·앰버서더)'인 팬덤: {n_dominant} / {len(ad_df)}")
print()
print("해당 팬덤:")
print(ad_df[ad_df["대표 메타요인이 브랜드·상업형인가"]][["팬덤", "구분", "브랜드·상업형 비중"]])


대표 메타요인(dominant_factor)이 '브랜드·상업형(광고·앰버서더)'인 팬덤: 1 / 100

해당 팬덤:
       팬덤  구분  브랜드·상업형 비중
1  사이먼도미닉  힙합      0.2553



## 4. 카테고리별 평균 비중 — 어느 장르가 "광고·브랜드" 성향이 강한가

100개 팬덤을 `구분`(category)별로 묶어 "브랜드·상업형" 평균 비중을 비교한다. K-pop
아이돌 그룹보다 트로트·힙합 솔로 아티스트 쪽이 오히려 높게 나오는지(광고·모델·앰버서더
활동이 도리어 개인 아티스트 브랜딩에서 더 자주 언급되는지) 실측으로 확인한다.


In [5]:

cat_df = (
    ad_df.groupby("구분")["브랜드·상업형 비중"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "평균 비중", "count": "팬덤 수"})
    .sort_values("평균 비중", ascending=False)
)
cat_df["평균 비중"] = cat_df["평균 비중"].round(4)
cat_df


,평균 비중,팬덤 수
구분,,
트로트,0.1420,9
힙합,0.1330,10
솔로,0.1064,18
K-pop 걸그룹,0.0853,23
발라드,0.0763,16
록,0.0748,2
K-pop 보이그룹,0.0736,15
혼성,0.0689,1
K-pop 보이그룹 /록,0.0592,3



## 5. 한계 (문서에서 이미 밝힌 것 재확인)

1. 이 노트북이 산출한 "브랜드·상업형 비중"은 원본 Ad/Commercial Index의 "광고비중"과
   **개념은 대응하지만 계산 방법이 다르다** — 원본은 업종 키워드 사전 매칭 기반 정수 카운트
   비율, 이 노트북은 LDA 토픽 혼합비중이다. 두 수치를 같은 척도로 비교하거나 원본 수치를
   역산하는 데 쓰면 안 된다.
2. **20개 업종 세분류는 전혀 복구되지 않았다** — "대표 메타요인"은 6개 메타요인 중 하나를
   가리킬 뿐, 원본의 "대표업종"(예: 화장품/식품/패션 등 실제 업종명)에 대응하지 않는다.
3. 코퍼스 규모(5,612건, round22)가 원본 v7 최종 리포트의 코퍼스 규모와 다르므로, 이 노트북의
   표는 round22 기준의 **독립적인 병행 산출물**이지 원본 `ad_commercial_index_v7.csv`의
   재현이 아니다.
